In [2]:
import base64
from io import BytesIO
from pdfplumber import open
from PIL import Image
from openai import OpenAI

# API密钥
API_KEY = "xxxxxxx"
# 本地PDF文件路径
PDF_PATH = "./files/test.pdf"

# 从PDF第一页提取图片
def extract_first_page_image(pdf_path):
    with open(pdf_path) as pdf:
        first_page = pdf.pages[0]
        # 将PDF页面渲染为图片
        img = first_page.to_image(resolution=300)
        # 保存到BytesIO
        img_byte_arr = BytesIO()
        img.save(img_byte_arr, format='PNG')
        img_byte_arr.seek(0)
        return img_byte_arr

# 主函数
def main():
    # 获取PDF第一页图片
    img_byte_arr = extract_first_page_image(PDF_PATH)
    # 将图片转换为base64编码
    base64_image = base64.b64encode(img_byte_arr.getvalue()).decode('utf-8')

    # 初始化OpenAI客户端
    client = OpenAI(
        # 如果没有配置环境变量，请用百炼API Key替换：api_key="sk-xxx"
        api_key = API_KEY,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )

    is_answering = False   # 判断是否结束思考过程并开始回复
    
    # 构造请求
    completion = client.chat.completions.create(
        model='qwen3-vl-plus',
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "请解析这张图片的内容。"
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        stream=True,
    )
    
    print("\n" + "=" * 20 + "思考过程" + "=" * 20 + "\n")

    for chunk in completion:
        # 如果chunk.choices为空，则打印usage
        if not chunk.choices:
            print("\nUsage:")
            print(chunk.usage)
        else:
            delta = chunk.choices[0].delta
            # 打印思考过程
            if hasattr(delta, 'reasoning_content') and delta.reasoning_content != None:
                print(delta.reasoning_content, end='', flush=True)
            else:
                # 开始回复
                if delta.content != "" and is_answering is False:
                    print("\n" + "=" * 20 + "完整回复" + "=" * 20 + "\n")
                    is_answering = True
                # 打印回复过程
                print(delta.content, end='', flush=True)

if __name__ == "__main__":
    main()


====================思考过程====================


====================完整回复====================

该图片是一道**2022年系统架构师考试真题（第1章：系统设计与建模，第2题）**，属于软件系统设计与数据流图（DFD）建模的综合分析题。以下是对图片内容的全面、结构化解析：

---

### 一、题目背景与目标
- **场景**：某能源企业拟开发“煤矿建设项目安全预警系统”，旨在保障从业人员生命安全。
- **核心任务**：根据系统功能需求（a）~（h），补全给定的数据流图（DFD）中编号为（1）~（6）的空白处理过程（即加工/变换节点），并简要说明该DFD在分层细化过程中所遵循的**数据平衡原则**。

---

### 二、系统功能列表（a）~（h）
| 编号 | 功能描述                     | 关键词               |
|------|------------------------------|----------------------|
| (a)  | 项目信息维护                 | 维护项目基础信息     |
| (b)  | 影响因素录入                 | 录入事故影响因素     |
| (c)  | 关联事故录入                 | 录入历史关联事故     |
| (d)  | 安全评价得分                 | 计算/生成安全评分    |
| (e)  | 项目指标预警分析             | 基于指标进行风险预警 |
| (f)  | 项目指标填报                 | 填报各类安全指标数据 |
| (g)  | 项目指标审核                 | 审核填报的指标数据   |
| (h)  | 项目指标确认                 | 确认最终指标有效性   |

> 注：这些功能是后续填空的依据。

---

### 三、数据流图（DFD）结构解析

图中包含：
- **外部实体（矩形）**：  
  - 项目管理员  
  - 安全员  
  - 安全副经理  
  - 项目经理  

- **数据